In [14]:
import requests
import tiktoken

# Download Tiny Shakespeare dataset
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
text = requests.get(url).text

enc = tiktoken.get_encoding('gpt2')
tokens = enc.encode(text)

import numpy as np
train_tokens = np.array(tokens[:int(len(tokens)*0.9)], dtype=np.uint16)
val_tokens = np.array(tokens[int(len(tokens)*0.9):], dtype=np.uint16)

train_tokens.tofile('train.bin')
val_tokens.tofile('val.bin')
print(f"Train tokens: {len(train_tokens)} | Val tokens: {len(val_tokens)}")

Train tokens: 304222 | Val tokens: 33803


In [12]:
import math
import torch
import torch.nn as nn
from torch.nn import functional as F

class Head(nn.Module):
    """ one head of self-attention """
    def __init__(self, head_size, n_embd, block_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.head_size = head_size

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B, T, hs)
        q = self.query(x) # (B, T, hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2, -1) * (self.head_size ** -0.5) # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B, T, hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """
    def __init__(self, num_heads, head_size, n_embd, block_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size, n_embd, block_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out

class FeedForward(nn.Module):
    """ a simple linear layer followed by a non-linearity """
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size, n_embd, block_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, block_size) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :] # focus on the last time step
            probs = F.softmax(logits, dim=-1) # (B, vocab_size)
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [16]:
import torch
import numpy as np

# --- Hyperparameters ---
batch_size = 32          # Increased for GPU efficiency on Colab!
block_size = 64          # What is the maximum context length for predictions?
max_iters = 1000         # Total training steps
eval_interval = 250
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 128
n_head = 4
n_layer = 4
vocab_size = 50257       # Standard GPT-2 vocab size from tiktoken

print(f"Using device: {device}")

# --- Load Data ---
train_data = np.memmap('train.bin', dtype=np.uint16, mode='r')
val_data = np.memmap('val.bin', dtype=np.uint16, mode='r')

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype('int64')) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype('int64')) for i in ix])
    return x.to(device), y.to(device)

# --- Model Initialization ---
model = MiniGPT(vocab_size, n_embd, n_head, n_layer, block_size)
m = model.to(device)

print(f"{sum(p.numel() for p in m.parameters())/1e6:.2f}M parameters in model.")

# --- Optimizer ---
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# --- Training Loop ---
for iter in range(max_iters):

    # Sample a batch of data
    xb, yb = get_batch('train')

    # Evaluate loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if iter % eval_interval == 0 or iter == max_iters - 1:
        print(f"step {iter}: train loss {loss.item():.4f}")

# Save the trained model weights
torch.save(model.state_dict(), 'mini_gpt_model.pt')
print("Training complete! Model saved to mini_gpt_model.pt")

Using device: cuda
13.72M parameters in model.
step 0: train loss 10.9728
step 250: train loss 5.8270
step 500: train loss 5.3261
step 750: train loss 4.8274
step 999: train loss 4.6623
Training complete! Model saved to mini_gpt_model.pt


In [17]:
import torch
import tiktoken

# --- Setup ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
enc = tiktoken.get_encoding('gpt2')

# Re-initialize model structure using the same hyperparameters
model = MiniGPT(vocab_size=50257, n_embd=128, n_head=4, n_layer=4, block_size=64)
model.load_state_dict(torch.load('mini_gpt_model.pt', map_location=device))
model = model.to(device)
model.eval()

# --- Generate Text ---
# Start with a newline token (or encode a prompt like "ROMEO:")
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_tokens = model.generate(context, max_new_tokens=500)[0].tolist()

decoded_text = enc.decode(generated_tokens)
print("--- Generated Output ---")
print(decoded_text)

--- Generated Output ---
!
3 KING payable, and thou Richard the sun
And my lords to come to-t'd doth.

Shepherd:
SICINI Const you have you
Of peace in London of the truth:
You met we fear of us remove on you being, but hour saw,
Come not well, but he must all to beiling a of Dorin
Go enough?

 wreckasters ere under captain shortly fear to circumstance;
Clcius!
BecUS:
If your wait, yes without censinks,iled it on, I'llretch
He of such done; then bear
For beingate read.

ISELI know you not pass to thy coin;
Commman:
there's king in repro I reward, I sw,
Inier ton's flowers's late hath beler fool.

Not three slew help.What peA:
My villain, I had my plethora,ule thee-learn was alone haste's I pray
For world bear me well straight.

ISENCE:
O, your heart to the trumpetel: pardon this king,
How, was roaring's pass'd ready stir,
And disperse their close.
To look the closely enemies this
Ay in true of greatove two sw great William,
Atis quite'd with begun
But had of his kindness, wasiss
 knejoy